In [ ]:
!pip install torch 
!pip install torchvision

In [ ]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(199)

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class CubeCylinderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
    
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        
        cube_path = os.path.join(root_dir, "fCube_Images")
        cylinder_path = os.path.join(root_dir, "fCylinder_Images")
        

        for file in os.listdir(cube_path):
            if file.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                self.samples.append(
                    (os.path.join(cube_path, file), 0)
                )
        
        for file in os.listdir(cylinder_path):
            if file.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                self.samples.append(
                    (os.path.join(cylinder_path, file), 1)
                )
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        return image, label
    
    

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

dataset = CubeCylinderDataset(root_dir=".", transform=transform)

print("Total de imagens:", len(dataset))
img, label = dataset[0]
print("Shape:", img.shape, "Label:", label)

In [ ]:
from SupervisedVAE import sGuidedVAE

from torch.utils.data import DataLoader, random_split

dataset_size = len(dataset)


train_size = int(0.8 * dataset_size)
test_size = dataset_size - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
def train_supervised(epoch, model, model_c,
                     optimizer, optimizer_c,
                     dataloader, w_internal, w_adv, device):

    model.train()
    model_c.train()

    re_loss = 0
    kld_loss = 0
    cls_internal_error = 0
    cls_aux_error = 0
    cls_adv_error = 0

    correct_internal = 0
    correct_aux = 0
    correct_adv = 0

    for batch_idx, (data, label) in enumerate(tqdm(dataloader, desc=f"Epoch {epoch}")):

        data = data.to(device)
        label = label.float().unsqueeze(1).to(device)

        # =====================================================
        # 1️⃣ TREINA VAE (recon + KL + classificador interno)
        # =====================================================

        optimizer.zero_grad()

        recon_batch, mu, logvar, cls_out = model(data)

        MSE = F.mse_loss(recon_batch, data, reduction="sum")
        MSE = F.l1_loss(recon_batch, data, reduction="sum")
        KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

        loss_cls_internal = F.binary_cross_entropy(cls_out, label, reduction='sum')

        loss_vae = MSE + KLD + w_internal * loss_cls_internal

        loss_vae.backward()
        optimizer.step()

        re_loss += MSE.item()
        kld_loss += KLD.item()
        cls_internal_error += loss_cls_internal.item()

        # =====================================================
        # 2️⃣ TREINA CLASSIFICADOR AUXILIAR
        # =====================================================

        optimizer_c.zero_grad()

        with torch.no_grad():
            mu, logvar = model.encode(data)

        z_aux = mu[:,1:]   # 🔑 usa MU (mais estável)

        cls1 = model_c(z_aux)

        loss_aux = F.binary_cross_entropy(cls1, label, reduction='sum')

        loss_aux.backward()
        optimizer_c.step()

        cls_aux_error += loss_aux.item()

        # =====================================================
        # 3️⃣ TREINA ENCODER ADVERSARIALMENTE
        # =====================================================

        optimizer.zero_grad()

        mu, logvar = model.encode(data)

        z_aux = mu[:,1:]   # 🔑 usa MU novamente

        cls2 = model_c(z_aux)

        neutral_label = torch.full_like(label, 0.5)

        loss_adv = F.binary_cross_entropy(cls2, neutral_label, reduction='sum')

        (w_adv * loss_adv).backward()
        optimizer.step()

        cls_adv_error += loss_adv.item()

        # =====================================================
        # Acurácias
        # =====================================================

        pred_internal = (cls_out > 0.5).float()
        correct_internal += pred_internal.eq(label).sum().item()

        pred_aux = (cls1 > 0.5).float()
        correct_aux += pred_aux.eq(label).sum().item()

        pred_adv = (cls2 > 0.5).float()
        correct_adv += pred_adv.eq(label).sum().item()

    # =====================================================
    # PRINT
    # =====================================================

    N = len(dataloader.dataset)

    print(f'''
====> Epoch {epoch}
Recon Loss: {re_loss/N:.4f} | KLD: {kld_loss/N:.4f}

Cls Internal:
Loss {cls_internal_error/N:.4f}
Acc  {100*correct_internal/N:.2f}%

Cls Aux (predict label from z[1:]):
Loss {cls_aux_error/N:.4f}
Acc  {100*correct_aux/N:.2f}%

Cls Adv (encoder confusion):
Loss {cls_adv_error/N:.4f}
Acc  {100*correct_adv/N:.2f}%
''')

In [ ]:
class LatentClassifier(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim-1, 32),  # remove primeira dimensão
            nn.ReLU(True),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, z):
        return self.net(z)


In [ ]:
from torch.optim import Adam

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

latent_dim = 4

model = sGuidedVAE(latent_dim=latent_dim).to(device)

optimizer = Adam(model.parameters(), lr=1e-3)

# pesos das losses
w_internal = 10.0
w_adv = 300.0

epochs = 50

for epoch in range(1, epochs + 1):

    train_supervised(
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        dataloader=train_loader,
        w_internal=w_internal,
        w_adv=w_adv,
        device=device
    )

In [ ]:
import matplotlib.pyplot as plt
import torch


def show_reconstructions(model, dataloader, device, n=8):
    model.eval()
    
    with torch.no_grad():
        data, label = next(iter(dataloader))
        data = data.to(device)

        recon, _, _, _ = model(data)

        data = data.cpu()
        recon = recon.cpu()

        fig, axes = plt.subplots(2, n, figsize=(2*n, 4))

        for i in range(n):
            # Original
            axes[0, i].imshow(data[i].permute(1, 2, 0))
            axes[0, i].set_title("Original")
            axes[0, i].axis("off")

            # Reconstrução
            axes[1, i].imshow(recon[i].permute(1, 2, 0))
            axes[1, i].set_title("Recon")
            axes[1, i].axis("off")

        plt.tight_layout()
        plt.show()

In [ ]:
show_reconstructions(model, test_loader, device, n=8)

In [ ]:
def extract_latents(model, dataloader, device):
    model.eval()

    cubes_latents = []
    cylinders_latents = []

    with torch.no_grad():
        for data, label in dataloader:

            data = data.to(device)
            label = label.to(device)

            mu, logvar = model.encode(data)

            for i in range(len(label)):
                if label[i] == 0:      # cubo
                    cubes_latents.append(mu[i].cpu())
                else:                  # cilindro
                    cylinders_latents.append(mu[i].cpu())

    cubes_latents = torch.stack(cubes_latents)
    cylinders_latents = torch.stack(cylinders_latents)

    return cubes_latents, cylinders_latents

In [ ]:
cubes_latents, cylinders_latents = extract_latents(model, test_loader, device)

print("Cubes:", cubes_latents.shape)
print("Cylinders:", cylinders_latents.shape)

cm = cubes_latents.mean(dim= 0)
cym = cylinders_latents.mean(dim = 0)

z1 = cm
z2 = cm 

z2 -= torch.tensor([-1*0.4054, 0, 0, 0, 0 ,0 ,0 ,0])

In [ ]:
import matplotlib.pyplot as plt
import torch

model.eval()

# média dos cubos (base latente)
base1 = cubes_latents.mean(dim=0)
base2 = torch.cat([
    cubes_latents.mean(dim=0)[:1],
    cylinders_latents.mean(dim=0)[1:]
])
# valores que vamos testar na dimensão 0
vals = torch.linspace(-1, 1, 25)

images = []

with torch.no_grad():

    for v in vals:
        z = base1.clone()
        z[0] = v     # varia a dimensão da classe

        z = z.unsqueeze(0).to(device)

        img = model.decode(z)[0].cpu()
        images.append(img)

In [ ]:
fig, axes = plt.subplots(5,5, figsize=(10,10))

for i, ax in enumerate(axes.flatten()):

    img = images[i].permute(1,2,0).numpy()

    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"{vals[i]:.2f}")

plt.suptitle("Variação da dimensão latente z0")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import torch

model.eval()

# média dos cubos (base latente)
base1 = cubes_latents.mean(dim=0)
base2 = torch.cat([
    cubes_latents.mean(dim=0)[:1],
    cylinders_latents.mean(dim=0)[1:]
])
# valores que vamos testar na dimensão 0
vals = torch.linspace(-1, 1, 25)

images = []

with torch.no_grad():

    for v in vals:
        z = base2.clone()
        z[0] = v     # varia a dimensão da classe

        z = z.unsqueeze(0).to(device)

        img = model.decode(z)[0].cpu()
        images.append(img)

In [ ]:
fig, axes = plt.subplots(5,5, figsize=(10,10))

for i, ax in enumerate(axes.flatten()):

    img = images[i].permute(1,2,0).numpy()

    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"{vals[i]:.2f}")

plt.suptitle("Variação da dimensão latente z0")
plt.show()